## Attention - But du fichier

Ce fichier **n'a pas pour objectif de réaliser un véritable fine-tuning**.  
Le but principal est simplement de **vérifier que tout le code d'entraînement, de création des embeddings et de recherche fonctionne correctement**.

Quelques points importants à noter :

1. Les modèles entraînés ici peuvent produire des résultats **moins bons que le modèle original pré-entraîné**.  
   - Cela est normal car le dataset utilisé est **très petit**, avec seulement quelques exemples.  
   - Les données ne sont **pas parfaitement adaptées au domaine du traitement du signal**.  
   - D'autres facteurs peuvent influencer les performances (taille limitée, diversité insuffisante, etc.).

2. Pour l’entraînement de ce fichier, nous avons utilisé un modèle **très simple et léger** : `all-MiniLM-L6-v2`.
   - Ce modèle est beaucoup moins performant que le modèle **bge-m3** ou le modèle **multilingual-e5-large**.  
   - Il a été choisi uniquement pour **vérifier le pipeline et la cohérence du code**, pas pour obtenir des performances optimales.

3. Les bases de données (datasets) utilisées ici sont **uniquement des exemples de vérification**, elles **ne doivent pas être utilisées pour un véritable entraînement sur le domaine du traitement du signal**.  

4. Ce fichier montre seulement que **le code fonctionne à 100 %**, que les embeddings se créent correctement, que les recherches vectorielles sont possibles et que le pipeline de test est opérationnel.

## Étape suivante : 

construire un dataset complet et de qualité, spécifiquement adapté au **traitement du signal**, pour entraîner réellement le modèle et évaluer ses performances dans ce domaine.

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 0 – Imports et Visualisation des datasets</h1>


In [8]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointIdsList, PointStruct, VectorParams
from sentence_transformers import InputExample, losses, SentenceTransformer
from torch.utils.data import DataLoader
from copy import deepcopy
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import pandas as pd
import json

In [9]:
df_original = pd.read_csv("original_data.csv", sep=";")

In [10]:
df_original.head(10)

,Chunk,Doc,Page
0,traitement du signal algorithme des libellules...,Traitement_du_Signal_pour_Télécommunications.pdf,1
1,rétropropagation,Théorie_des_Ondelettes_et_Applications.pdf,2
2,pyramide orientable,Traitement_du_Signal_Numérique.pdf,3
3,traitement du signal algorithme évolutionnaire...,Séparation_aveugle_de_Sources.pdf,2
4,wavelet-based signal processing in multi-objec...,Machine_Learning_for_Signal_Analysis.pdf,3
5,wavelet-based signal processing in multi-objec...,Audio_Signal_Processing_Methods.pdf,3
6,wavelet-based signal processing in multi-objec...,Biomedical_Signal_Processing.pdf,1
7,traitement du signal recuit simulé par ondelettes,Algorithmes_Adaptatifs_en_Traitement_du_Signal...,1
8,séries temporelles,Séparation_aveugle_de_Sources.pdf,3
9,traitement du signal optimisateur des loups gr...,Méthodes_Modernes_en_Traitement_dImage.pdf,3


In [11]:
df_training = pd.read_csv("training_data.csv", sep=",")

In [12]:
df_training.head(10)

,text1,text2,similarity_score,language_pair
0,traitement sonar,traitement radar,0.789518,fr-fr
1,analyse spectrale,autocorrélation,0.836305,fr-fr
2,diagramme de Bode,fréquence d'échantillonnage,0.851264,fr-fr
3,communications numériques,traitement vocal,0.799205,fr-fr
4,linéarité,bande passante,0.755860,fr-fr
5,égalisation de canal,communications numériques,0.752648,fr-fr
6,fréquence d'échantillonnage,linéarité,0.798378,fr-fr
7,pôles et zéros,fonction de transfert,0.994506,fr-fr
8,réponse en fréquence,fréquence de Nyquist,0.774414,fr-fr
9,reconnaissance de formes,modulation QAM,0.834021,fr-fr


In [13]:
def charger_json_simple(chemin_fichier):
    """Charge simplement un fichier JSON"""
    with open(chemin_fichier, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

df_test = charger_json_simple("./test_data.json")
queries = df_test['queries']
corpus = df_test['corpus']
ground_truth = df_test['ground_truth']

queries

[{'id': 'q1',
  'text': "Comment la transformée de Fourier rapide (FFT) améliore-t-elle l'efficacité du calcul spectral par rapport à la transformée de Fourier discrète standard ?"},
 {'id': 'q2',
  'text': 'Quelles sont les méthodes les plus efficaces pour réduire le bruit blanc gaussien dans les signaux EEG tout en préservant les composantes neurologiques importantes ?'},
 {'id': 'q3',
  'text': 'Comment les coefficients cepstraux MFCC sont-ils calculés et pourquoi sont-ils si efficaces pour la reconnaissance automatique de la parole ?'},
 {'id': 'q4',
  'text': "Quels sont les avantages et limitations des ondelettes de Daubechies pour l'analyse multi-résolution des signaux non stationnaires ?"},
 {'id': 'q5',
  'text': "Comment les filtres de Kalman sont-ils utilisés pour la prédiction et l'estimation dans les systèmes de tracking radar ?"},
 {'id': 'q6',
  'text': 'Quelles techniques de traitement du signal permettent de détecter les arythmies cardiaques dans les signaux ECG de lon

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 1 – Créer une collection</h1>


In [14]:
client = QdrantClient("localhost", port=6333)

client.recreate_collection(
    collection_name="documents_ts",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # size=384 pour all-MiniLM-L6-v2 et size=1024 pour BAAI/bge-m3 et intfloat/multilingual-e5-large
)

/tmp/ipykernel_134125/2884627132.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 2 – Ajouter des embeddings</h1>


In [15]:
# Modèle pour embeddings
original_model = SentenceTransformer("all-MiniLM-L6-v2")  # ou ["BAAI/bge-m3", "intfloat/multilingual-e5-large", "all-MiniLM-L6-v2"]

# Calcul des embeddings et préparation pour Qdrant
points = []
for idx, row in df_original.iterrows():
    try:
        # Encoder le texte
        emb = original_model.encode(row["Chunk"]).tolist()
        
        # Créer le point pour Qdrant
        point = PointStruct(
            id=idx,
            vector=emb,
            payload={
                "Chunk": row["Chunk"],
                "Doc": row["Doc"],
                "Page": int(row["Page"])  # Convertir en entier
            }
        )
        points.append(point)
        
        # Afficher la progression
        if (idx + 1) % 100 == 0:
            print(f"Traitement de {idx + 1}/{len(df_original)} éléments...")
            
    except Exception as e:
        print(f"Erreur avec l'élément {idx}: {e}")
        continue

# Upsert dans Qdrant
try:
    result = client.upsert(
        collection_name="documents_ts",
        points=points
    )
    
    print(f"Tous les chunks ont été insérés avec succès dans Qdrant !")
    
except Exception as e:
    print(f"Erreur lors de l'insertion dans Qdrant: {e}")

Traitement de 100/818 éléments...
Traitement de 200/818 éléments...
Traitement de 300/818 éléments...
Traitement de 400/818 éléments...
Traitement de 500/818 éléments...
Traitement de 600/818 éléments...
Traitement de 700/818 éléments...
Traitement de 800/818 éléments...
Tous les chunks ont été insérés avec succès dans Qdrant !


<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 3 – Rechercher</h1>


In [16]:
query = "Comment réduire le bruit dans un signal radar ?"
query_emb = original_model.encode(query).tolist()

hits = client.search(
    collection_name="documents_ts",
    query_vector=query_emb,
    limit=3
)

for hit in hits:
    print(hit.payload["Chunk"], " (score:", hit.score, ")")

traitement du signal radar par ondelettes  (score: 0.7009159 )
traitement du signal surveillance d'état par ondelettes  (score: 0.59994894 )
traitement du signal informatique en brouillard par ondelettes  (score: 0.5951952 )


/tmp/ipykernel_134125/2433353422.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 4 – Visualiser les vecteurs stockés</h1>


In [17]:
# Récupérer les points AVEC vecteurs
points, next_page = client.scroll(
    collection_name="documents_ts",
    limit=10,
    with_vectors=True  # <-- important
)

for point in points:
    print("ID:", point.id)
    if point.vector is not None:
        print("Vector (d'abord 10 valeurs):", point.vector[:10])
    else:
        print("Vector: None")
    print("Payload:", point.payload)
    print("-" * 50)

ID: 0
Vector (d'abord 10 valeurs): [-0.04996257, -0.07562201, 0.0052525904, -0.11592169, -0.04092223, 0.009504886, 0.0042218603, -0.020785956, -0.008901179, -0.016436186]
Payload: {'Chunk': 'traitement du signal algorithme des libellules multi-objectif par ondelettes', 'Doc': 'Traitement_du_Signal_pour_Télécommunications.pdf', 'Page': 1}
--------------------------------------------------
ID: 1
Vector (d'abord 10 valeurs): [-0.070232876, -0.04438213, 0.07804057, 0.018153237, -0.017100872, 0.07325364, -0.0001110827, -0.055199124, -0.03014075, -0.02658385]
Payload: {'Chunk': 'rétropropagation', 'Doc': 'Théorie_des_Ondelettes_et_Applications.pdf', 'Page': 2}
--------------------------------------------------
ID: 2
Vector (d'abord 10 valeurs): [-0.013974313, 0.055454437, -0.026813736, -0.038679063, -0.09254562, 0.071276024, 0.005469669, -0.00875556, -0.032210473, 0.020617018]
Payload: {'Chunk': 'pyramide orientable', 'Doc': 'Traitement_du_Signal_Numérique.pdf', 'Page': 3}
------------------

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 5 – Fine tuning léger</h1>


In [18]:
train_examples = []

for _, row in df_training.iterrows():
    example = InputExample(
        texts=[row['text1'], row['text2']],
        label=float(row['similarity_score'])
    )
    train_examples.append(example)

In [19]:
train_loss = losses.CosineSimilarityLoss(model=original_model)

In [20]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [21]:
# Fine-tuning léger
trained_model = deepcopy(original_model)

trained_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="./fine_tuned_all_MiniLM_L6_v2"
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/home/saad/keras_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 6 – Rechargement du nouveau modèle</h1>


In [22]:
trained_model = SentenceTransformer("./fine_tuned_all_MiniLM_L6_v2")

##### Réajouter les embeddings

In [23]:
# Calcul des embeddings et préparation pour Qdrant
points = []
for idx, row in df_original.iterrows():
    try:
        # Encoder le texte
        emb = trained_model.encode(row["Chunk"]).tolist()
        
        # Créer le point pour Qdrant
        point = PointStruct(
            id=idx,
            vector=emb,
            payload={
                "Chunk": row["Chunk"],
                "Doc": row["Doc"],
                "Page": int(row["Page"])  # Convertir en entier
            }
        )
        points.append(point)
        
        # Afficher la progression
        if (idx + 1) % 100 == 0:
            print(f"Traitement de {idx + 1}/{len(df_original)} éléments...")
            
    except Exception as e:
        print(f"Erreur avec l'élément {idx}: {e}")
        continue

# Upsert dans Qdrant
try:
    result = client.upsert(
        collection_name="documents_ts",
        points=points
    )
    
    print(f"Tous les chunks ont été insérés avec succès dans Qdrant !")
    
except Exception as e:
    print(f"Erreur lors de l'insertion dans Qdrant: {e}")

Traitement de 100/818 éléments...
Traitement de 200/818 éléments...
Traitement de 300/818 éléments...
Traitement de 400/818 éléments...
Traitement de 500/818 éléments...
Traitement de 600/818 éléments...
Traitement de 700/818 éléments...
Traitement de 800/818 éléments...
Tous les chunks ont été insérés avec succès dans Qdrant !


##### Rechrcher

In [24]:
query = "Comment réduire le bruit dans un signal radar ?"
query_emb = trained_model.encode(query).tolist()

hits = client.search(
    collection_name="documents_ts",
    query_vector=query_emb,
    limit=3
)

for hit in hits:
    print(hit.payload["Chunk"], " (score:", hit.score, ")")

traitement du signal logistique par ondelettes  (score: 0.83899546 )
traitement du signal relativité par ondelettes  (score: 0.83763695 )
traitement du signal pêches par ondelettes  (score: 0.8346449 )


/tmp/ipykernel_134125/3940075080.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


##### Remarque:
Le modèle fine-tuné retourne maintenant des vecteurs qui sont plus proches de la requête dans le domaine traitement du signal, donc il a appris quelque chose sur le domaine.

<h1 style="background-color: gray;
           color: black;
           padding: 20px;
           text-align: center;">Étape 7 – Test du modèle</h1>


### Évaluation du modèle d’embeddings

Pour évaluer la qualité d’un modèle d’embeddings dans une tâche de **retrieval** (recherche de documents pertinents à partir d’une requête), on utilise plusieurs métriques standards :  
**MRR (Mean Reciprocal Rank)**, **Recall@k**, et **nDCG (Normalized Discounted Cumulative Gain)**.  

---

#### 1. Mean Reciprocal Rank (MRR)

Le **MRR** mesure la capacité du modèle à placer le premier document pertinent le plus haut possible dans le classement.

- Pour chaque requête, on calcule le **rang** (*rank*) du premier document pertinent dans la liste triée des résultats.  
- On prend ensuite son **inverse** (1/rank).  
- La moyenne de ces valeurs sur toutes les requêtes donne le **MRR**.

$$
MRR = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i}
$$

> **Interprétation :**  
> - Un MRR proche de **1.0** signifie que les documents pertinents apparaissent presque toujours tout en haut du classement.  
> - Plus le MRR est élevé, meilleure est la précision du modèle sur les premiers résultats.

---

#### 2. Recall@k

Le **Recall@k** mesure la proportion de documents pertinents retrouvés **parmi les k premiers résultats**.

$$
Recall@k = \frac{\text{documents pertinents retrouvés dans le top-k}}{\text{documents pertinents totaux}}
$$

> **Interprétation :**  
> - Si `Recall@10 = 0.8`, cela signifie que 80 % des documents pertinents se trouvent parmi les 10 premiers résultats.  
> - Cette métrique évalue la **capacité de couverture** du modèle (retrouver tous les bons documents).

---

#### 3. Normalized Discounted Cumulative Gain (nDCG)

Le **nDCG** mesure la qualité **du classement** des documents pertinents.  
Il prend en compte la position des documents pertinents et attribue une **pondération logarithmique** : les documents retrouvés plus haut valent plus.

$$
nDCG@k = \frac{DCG@k}{IDCG@k}
$$
avec  
$$
DCG@k = \sum_{i=1}^{k} \frac{rel_i}{\log_2(i+1)}
$$
et `rel_i` est la pertinence du document à la position *i*.  

> **Interprétation :**  
> - nDCG varie entre **0 et 1**.  
> - Plus il est proche de 1, plus les documents pertinents sont bien classés dans les premières positions.  
> - Cette métrique reflète la **qualité du tri des résultats**.

---

#### En résumé

| Métrique | Mesure | Ce qu’elle indique | Valeur idéale |
|-----------|---------|--------------------|----------------|
| **MRR** | Position du 1er document pertinent | Rapidité à trouver un résultat pertinent | Proche de 1 |
| **Recall@k** | Taux de documents pertinents retrouvés dans le top-k | Couverture des résultats | Proche de 1 |
| **nDCG@k** | Pertinence pondérée par la position | Qualité du classement global | Proche de 1 |

---

**En pratique :**  
- On utilise **MRR** pour évaluer la **précision des premiers résultats**,  
- **Recall@k** pour mesurer la **capacité à tout retrouver**,  
- et **nDCG** pour juger la **qualité du classement global**.